### 要求
整个⽣命周期内的最⼤内存使⽤量低于 80MB
你不能牺牲太多的时间性能来最⼩化你的内存使⽤量。程序需要在 60 秒内
完成运⾏。

用快速幂算法呢，加快收敛
### 优化方向-减少内存
1. 块矩阵
2. 稀疏矩阵https://zhuanlan.zhihu.com/p/557231877
3. 其他方法

### 问题
1. 死胡同问题
   
   某个节点只有指向自己的链接或者没有连接咋办
   
2. 蜘蛛网问题
   
   部分子图为单独的联通分支，导致这部分子图偏高
   
   resolution：
   
   (1 - damping) * teleport 是为了防止蜘蛛网问题，让它有一定概率随机走到任意节点，跑出蜘蛛网

In [19]:
import numpy as np
from memory_profiler import profile


def build_graph(file_path):
    # 读取文件并构建节点集与边列表
    edges = []
    nodes = set()
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 2:
                print("Invalid line format:", line)
                continue
            u, v = parts
            edges.append((u, v))
            # 添加节点到集合中
            nodes.update([u, v])
    # 排序节点并建立索引
    nodes = sorted(list(nodes))
    node_index = {node: idx for idx, node in enumerate(nodes)}
    return edges, nodes, node_index

# 初始化转移矩阵
def build_transition_matrix(edges, nodes, node_index):
    N = len(nodes)# 节点个数
    M = np.zeros((N, N))
    # 构建转移矩阵 M，行表示出发点，列表示终点
    for u, v in edges:
        i = node_index[u]
        j = node_index[v]
        M[i, j] = 1
    # 归一化：将每一行除以该行的和，注意处理死节点（没有出边的节点）
    for i in range(N):
        row_sum = M[i].sum()
        if row_sum != 0:
            M[i] /= row_sum
        else:
            # 如果没有出边，则均匀分布到所有节点上
            # 黑洞！！！
            M[i] = np.ones(N) / N
    return M.T  # 转置后方便用列向量表示概率

def pagerank(M, damping=0.85, tol=1e-6, max_iter=10000):
    N = M.shape[0]
    # 初始化概率向量
    pr = np.ones(N) / N
    teleport = np.ones(N) / N
    for _ in range(max_iter):
        # (1 - damping) * teleport 是为了防止蜘蛛网问题，让它有一定概率随机走到任意节点，跑出蜘蛛网
        pr_new = damping * (M @ pr) + (1 - damping) * teleport
        if np.linalg.norm(pr_new - pr, ord=1) < tol:
            print(f"迭代第{_}次之后，PageRank 收敛")
            return pr_new
        pr = pr_new
    return pr

def main():
    file_path = "Data.txt"
    edges, nodes, node_index = build_graph(file_path)
    M = build_transition_matrix(edges, nodes, node_index)
    pr_values = pagerank(M)
    
    # 将节点与对应的 PageRank 值组合成元组列表
    node_pr = list(zip(nodes, pr_values))
    # 按照 PageRank 值降序排序
    sorted_node_pr = sorted(node_pr, key=lambda x: x[1], reverse=True)
    # 输出排序后的节点和 PageRank 值
    # for node, pr in sorted_node_pr:
    #     print("节点 {} 的 PageRank 值为: {:.6f}".format(node, pr))
    
    for i in range(100):
        node, pr = sorted_node_pr[i]
        print("节点 {} 的 PageRank 值为: {:.6f}".format(node, pr))
    print(pr_values.sum())
    # 将 Top-100 节点及其分数写入文件
    with open("Res.txt", "w") as f:
        for i in range(100):
            node, pr = sorted_node_pr[i]
            f.write(f"{node} {pr:.6f}\n")



In [20]:
main()

迭代第8次之后，PageRank 收敛
节点 286 的 PageRank 值为: 0.000198
节点 3473 的 PageRank 值为: 0.000196
节点 4951 的 PageRank 值为: 0.000192
节点 3890 的 PageRank 值为: 0.000191
节点 7365 的 PageRank 值为: 0.000189
节点 6359 的 PageRank 值为: 0.000188
节点 4352 的 PageRank 值为: 0.000182
节点 7032 的 PageRank 值为: 0.000181
节点 7541 的 PageRank 值为: 0.000181
节点 3699 的 PageRank 值为: 0.000180
节点 4877 的 PageRank 值为: 0.000179
节点 3242 的 PageRank 值为: 0.000177
节点 6503 的 PageRank 值为: 0.000176
节点 4221 的 PageRank 值为: 0.000175
节点 7293 的 PageRank 值为: 0.000174
节点 2276 的 PageRank 值为: 0.000174
节点 1189 的 PageRank 值为: 0.000174
节点 1441 的 PageRank 值为: 0.000172
节点 1866 的 PageRank 值为: 0.000171
节点 8602 的 PageRank 值为: 0.000170
节点 5012 的 PageRank 值为: 0.000170
节点 2342 的 PageRank 值为: 0.000170
节点 4660 的 PageRank 值为: 0.000170
节点 7353 的 PageRank 值为: 0.000169
节点 1237 的 PageRank 值为: 0.000169
节点 257 的 PageRank 值为: 0.000169
节点 8036 的 PageRank 值为: 0.000169
节点 8314 的 PageRank 值为: 0.000169
节点 7633 的 PageRank 值为: 0.000169
节点 2797 的 PageRank 值为: 0.000168
节点 6901 的 PageRank 值为:

In [3]:
import numpy as np
from scipy import sparse
from memory_profiler import profile


def build_graph(file_path):
    # 读取文件并构建节点集与边列表
    edges = []
    nodes = set()
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 2:
                print("Invalid line format:", line)
                continue
            u, v = parts
            edges.append((u, v))
            # 添加节点到集合中
            nodes.update([u, v])
    # 排序节点并建立索引
    nodes = sorted(list(nodes))
    node_index = {node: idx for idx, node in enumerate(nodes)}
    return edges, nodes, node_index

def build_transition_matrix(edges, nodes, node_index):
    N = len(nodes)
    # 使用COO格式初始化稀疏矩阵（适合构建阶段）
    row_indices = []
    col_indices = []
    data = []
    
    # 统计每个节点的出度
    out_degrees = np.zeros(N)
    for u, v in edges:
        i = node_index[u]
        j = node_index[v]
        out_degrees[i] += 1
    
    # 构建稀疏矩阵的数据
    for u, v in edges:
        i = node_index[u]
        j = node_index[v]
        if out_degrees[i] > 0:
            row_indices.append(j)  # 转置后的行（原列）
            col_indices.append(i)  # 转置后的列（原行）
            data.append(1.0 / out_degrees[i])
    
    # # 处理死节点（没有出边的节点）
    # for i in range(N):
    #     if out_degrees[i] == 0:
    #         for j in range(N):
    #             row_indices.append(j)
    #             col_indices.append(i)
    #             data.append(1.0 / N)
    
    # 创建稀疏矩阵（已转置）
    M = sparse.csr_matrix((data, (row_indices, col_indices)), shape=(N, N))
    return M

def pagerank(M, damping=0.85, tol=1e-6, max_iter=10000):
    N = M.shape[0]
    # 初始化概率向量
    pr = np.ones(N, dtype=np.float32) / N
    teleport = np.ones(N, dtype=np.float32) / N
    
    for iter_count in range(max_iter):
        # 稀疏矩阵乘法
        pr_new = damping * (M @ pr) + (1 - damping) * teleport * sum(pr)
        
        # 计算收敛程度
        err = np.linalg.norm(pr_new - pr, ord=1)
        pr = pr_new
        
        if err < tol:
            print(f"迭代第{iter_count}次之后，PageRank 收敛")
            return pr
            
    print(f"警告：达到最大迭代次数{max_iter}，但未收敛")
    return pr

@profile
def main():
    file_path = "../Data.txt"
    edges, nodes, node_index = build_graph(file_path)
    M = build_transition_matrix(edges, nodes, node_index)
    pr_values = pagerank(M)
    
    # 将节点与对应的 PageRank 值组合成元组列表
    node_pr = list(zip(nodes, pr_values))
    # 按照 PageRank 值降序排序
    sorted_node_pr = sorted(node_pr, key=lambda x: x[1], reverse=True)
    # 输出排序后的节点和 PageRank 值
    # for node, pr in sorted_node_pr:
    #     print("节点 {} 的 PageRank 值为: {:.6f}".format(node, pr))
    
    for i in range(100):
        node, pr = sorted_node_pr[i]
        print("节点 {} 的 PageRank 值为: {:.6f}".format(node, pr))
    print(pr_values.sum())
    
    # 将 Top-100 节点及其分数写入文件
    with open("Res2.txt", "w") as f:
        for i in range(100):
            node, pr = sorted_node_pr[i]
            f.write(f"{node} {pr:.15e}\n")
    
if __name__ == "__main__":
    main()



ERROR: Could not find file /var/folders/1n/4skwfl_n6gq3hxvc100399q80000gn/T/ipykernel_52850/3616139564.py
迭代第122次之后，PageRank 收敛
节点 286 的 PageRank 值为: 0.000000
节点 3473 的 PageRank 值为: 0.000000
节点 4951 的 PageRank 值为: 0.000000
节点 3890 的 PageRank 值为: 0.000000
节点 7365 的 PageRank 值为: 0.000000
节点 6359 的 PageRank 值为: 0.000000
节点 3699 的 PageRank 值为: 0.000000
节点 4352 的 PageRank 值为: 0.000000
节点 7032 的 PageRank 值为: 0.000000
节点 7541 的 PageRank 值为: 0.000000
节点 4877 的 PageRank 值为: 0.000000
节点 3242 的 PageRank 值为: 0.000000
节点 6503 的 PageRank 值为: 0.000000
节点 7293 的 PageRank 值为: 0.000000
节点 2276 的 PageRank 值为: 0.000000
节点 4221 的 PageRank 值为: 0.000000
节点 1189 的 PageRank 值为: 0.000000
节点 1441 的 PageRank 值为: 0.000000
节点 5012 的 PageRank 值为: 0.000000
节点 2342 的 PageRank 值为: 0.000000
节点 1866 的 PageRank 值为: 0.000000
节点 8602 的 PageRank 值为: 0.000000
节点 7353 的 PageRank 值为: 0.000000
节点 8036 的 PageRank 值为: 0.000000
节点 4660 的 PageRank 值为: 0.000000
节点 7633 的 PageRank 值为: 0.000000
节点 8314 的 PageRank 值为: 0.000000
节点 257 的 